# 01 — ETL
Caricamento, pulizia e salvataggio dei dati intermedi.

**Output:** `movies_cleaned.csv`, `edges.csv`, `collab_df.csv`, `node_features.csv`

In [3]:
import ast
import re
import unicodedata
import warnings
import random
import pandas as pd
import numpy as np
from collections import Counter
from itertools import combinations

warnings.filterwarnings('ignore')

## 1. Caricamento

In [4]:
df = pd.read_csv('dataset_ridotto.csv')
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
print(f"Righe originali: {len(df)}")

Righe originali: 8816


## 2. Parse colonne lista

In [5]:
def safe_parse(val):
    """Converte stringa tipo "['a', 'b']" in lista Python."""
    try:
        result = ast.literal_eval(val)
        return result if isinstance(result, list) else []
    except Exception:
        return []

df["Director_list"] = df["Director"].apply(safe_parse)
df["Stars_list"]    = df["Stars"].apply(safe_parse)
df["Genre_list"]    = df["Genre"].apply(safe_parse)

## 3. Pulizia

In [6]:
# Anno come numerico
df["Year of Release"] = pd.to_numeric(df["Year of Release"], errors="coerce")

# Filtro runtime
df = df[df["Run Time in minutes"].between(40, 600)]

# Deduplicazione
df = df.drop_duplicates(subset=["Movie Name", "Year of Release"])
print(f"Righe dopo filtro runtime e dedup: {len(df)}")

Righe dopo filtro runtime e dedup: 8815


## 4. Normalizzazione nomi

In [7]:
def normalize_name(name):
    """Rimuove accenti, normalizza spazi e applica title case."""
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = name.strip().title()
    name = re.sub(r"\s+", " ", name)
    return name

df["Stars_list"]    = df["Stars_list"].apply(lambda l: [normalize_name(x) for x in l])
df["Director_list"] = df["Director_list"].apply(lambda l: [normalize_name(x) for x in l])
df["Genre_list"]    = df["Genre_list"].apply(lambda l: [x.strip() for x in l])

## 5. Filtro rumore

In [8]:
NOISE = {"", "N/A", "Unknown", "None", "-"}

df["Stars_list"]    = df["Stars_list"].apply(
    lambda l: [x for x in l if x not in NOISE and len(x) > 2]
)
df["Director_list"] = df["Director_list"].apply(
    lambda l: [x for x in l if x not in NOISE and len(x) > 2]
)

df = df[df["Stars_list"].map(len) > 0]
df = df[df["Director_list"].map(len) > 0]
print(f"Righe dopo filtro nomi: {len(df)}")

Righe dopo filtro nomi: 8810


## 6. Edge list

In [9]:
edges = []
for _, row in df.iterrows():
    film = (row["Movie Name"], row["Year of Release"])
    for actor in row["Stars_list"]:
        edges.append({"person": actor, "film": str(film), "role": "actor"})
    for director in row["Director_list"]:
        edges.append({"person": director, "film": str(film), "role": "director"})

edge_df = pd.DataFrame(edges)
print(f"Edge list creata: {len(edge_df):,} archi")
print(f"Persone uniche (attori + registi): {edge_df['person'].nunique():,}")
print(f"  di cui attori:   {edge_df[edge_df['role']=='actor']['person'].nunique():,}")
print(f"  di cui registi:  {edge_df[edge_df['role']=='director']['person'].nunique():,}")
print(f"Film unici:        {edge_df['film'].nunique():,}")

Edge list creata: 44,804 archi
Persone uniche (attori + registi): 16,803
  di cui attori:   13,210
  di cui registi:  3,962
Film unici:        8,810


## 7. Check isolati

In [10]:
film_count = edge_df.groupby("person")["film"].nunique()
singletons = (film_count == 1).sum()
pct = singletons / len(film_count) * 100
print(f"Persone con 1 solo film (isolati potenziali): {singletons:,} ({pct:.1f}%)")
if pct > 60:
    print("⚠️  Oltre il 60% — considera di filtrare gli isolati prima della SNA")
else:
    print("✓  Sotto il 60% — rete sufficientemente connessa")

Persone con 1 solo film (isolati potenziali): 10,644 (63.3%)
⚠️  Oltre il 60% — considera di filtrare gli isolati prima della SNA


## 8. Collaborazioni tra attori

In [11]:
collab_counter = Counter()
for _, row in df.iterrows():
    for a, b in combinations(sorted(row["Stars_list"]), 2):
        collab_counter[(a, b)] += 1

collab_df = pd.DataFrame(
    [(a, b, w) for (a, b), w in collab_counter.items()],
    columns=["actor_a", "actor_b", "weight"]
).sort_values("weight", ascending=False)

print(f"Coppie totali: {len(collab_df):,}")
print("Top 10 coppie:")
print(collab_df.head(10).to_string(index=False))

Coppie totali: 50,404
Top 10 coppie:
             actor_a        actor_b  weight
        Adam Sandler    Kevin James       9
    Daniel Radcliffe   Rupert Grint       8
    Daniel Radcliffe    Emma Watson       7
         Emma Watson   Rupert Grint       7
  Megumi Hayashibara   Megumi Ogata       6
        Rani Mukerji Shah Rukh Khan       6
               Kajol Shah Rukh Khan       6
  Michelle Rodriguez     Vin Diesel       6
         Mahesh Babu    Prakash Raj       6
Helena Bonham Carter    Johnny Depp       6


## 9. Node features

In [12]:
actor_stats = []
for _, row in df.iterrows():
    for actor in row["Stars_list"]:
        actor_stats.append({
            "person": actor,
            "rating": row["Movie Rating"],
            "genre": row["Genre_list"][0] if row["Genre_list"] else "Unknown"
        })

node_metadata = pd.DataFrame(actor_stats)
node_features = node_metadata.groupby("person").agg(
    rating=("rating", "mean"),
    genre=("genre", lambda x: x.value_counts().index[0]),
    movie_count=("person", "count")
).reset_index()

## 10. Salvataggio

In [13]:
# Salva df con liste come stringhe JSON-safe
import ast
df_save = df.copy()
df_save["Stars_list"]    = df_save["Stars_list"].apply(str)
df_save["Director_list"] = df_save["Director_list"].apply(str)
df_save["Genre_list"]    = df_save["Genre_list"].apply(str)
df_save.to_csv("movies_cleaned.csv", index=False)

edge_df.to_csv("edges.csv", index=False)
collab_df.to_csv("collab_df.csv", index=False)
node_features.to_csv("node_features.csv", index=False)

print("✓ ETL completato")
print(f"  movies_cleaned.csv  — {len(df_save):,} righe")
print(f"  edges.csv           — {len(edge_df):,} righe")
print(f"  collab_df.csv       — {len(collab_df):,} righe")
print(f"  node_features.csv   — {len(node_features):,} righe")

✓ ETL completato
  movies_cleaned.csv  — 8,810 righe
  edges.csv           — 44,804 righe
  collab_df.csv       — 50,404 righe
  node_features.csv   — 13,210 righe
